# Phased Allocation with Handover — via API

Same scenario as `optimizer/example/phased_allocation_handover.ipynb`, but driven entirely through the REST API.

10 employees, project **gamma** split into two phases:

| Phase    | Slots | Skill emphasis      |
|----------|-------|---------------------|
| modeling | 3     | machine_learning    |
| analysis | 3     | data_science        |

With `handover=0` each phase is solved independently — the teams churn completely. With `handover=0.4` the phases are solved jointly and retaining a person earns a bonus.

> **Prerequisites:** the API server must be running on `http://localhost:8000`.
> From the repo root: `uv run uvicorn api.main:app --reload --app-dir backend/src`

In [2]:
import requests

BASE = "http://localhost:8000/api/v1"

resp = requests.get("http://localhost:8000/health")
resp.raise_for_status()
print(resp.json())

{'status': 'ok'}


## Create employees

Skills scored 0–5. ML engineers have a strong `machine_learning` skill; data scientists have a strong `data_science` skill. A few people (`ml_3`, `ds_5`) are dual-skilled — the handover weight will profitably retain those across phases.

In [3]:
ML  = "machine_learning"
DS  = "data_science"
DEV = "software_development"

people_payloads = [
    # ML engineers
    {"name": "ml_1", "role": "ML Engineer",       "seniority": "senior", "years_of_experience": 6, "skills": [{"id": ML, "level": 4.5}, {"id": DS, "level": 2.0}]},
    {"name": "ml_2", "role": "ML Engineer",       "seniority": "mid",    "years_of_experience": 4, "skills": [{"id": ML, "level": 4.0}, {"id": DS, "level": 1.5}]},
    {"name": "ml_3", "role": "ML Engineer",       "seniority": "lead",   "years_of_experience": 8, "skills": [{"id": ML, "level": 5.0}, {"id": DS, "level": 3.0}]},
    # Data scientists
    {"name": "ds_1", "role": "Data Scientist",    "seniority": "senior", "years_of_experience": 5, "skills": [{"id": DS, "level": 4.5}, {"id": ML, "level": 2.5}]},
    {"name": "ds_2", "role": "Data Scientist",    "seniority": "mid",    "years_of_experience": 3, "skills": [{"id": DS, "level": 4.0}, {"id": ML, "level": 1.5}]},
    {"name": "ds_3", "role": "Data Scientist",    "seniority": "senior", "years_of_experience": 7, "skills": [{"id": DS, "level": 5.0}, {"id": ML, "level": 2.0}]},
    {"name": "ds_4", "role": "Data Scientist",    "seniority": "junior", "years_of_experience": 2, "skills": [{"id": DS, "level": 3.5}, {"id": ML, "level": 1.0}]},
    {"name": "ds_5", "role": "Data Scientist",    "seniority": "lead",   "years_of_experience": 9, "skills": [{"id": DS, "level": 4.5}, {"id": ML, "level": 3.0}]},
    # Developers
    {"name": "dev_1", "role": "Software Developer", "seniority": "senior", "years_of_experience": 5, "skills": [{"id": DEV, "level": 4.5}]},
    {"name": "dev_2", "role": "Software Developer", "seniority": "mid",    "years_of_experience": 3, "skills": [{"id": DEV, "level": 4.0}]},
]

created_people = []
for payload in people_payloads:
    r = requests.post(f"{BASE}/people", json=payload)
    r.raise_for_status()
    created_people.append(r.json())

# Map name → API-assigned id for display later
id_to_name = {p["id"]: p["name"] for p in created_people}
print(f"Created {len(created_people)} people")

Created 10 people


## Create project

Project **gamma** with two phases. The top-level `n_slots` is overridden by the per-phase slot counts when `phases` is non-empty.

In [4]:
project_payload = {
    "name": "gamma",
    "description": "Two-phase project: ML modeling then data analysis.",
    "n_slots": 3,
    "phases": [
        {
            "id": "modeling",
            "n_slots": 3,
            "skill_requirements": [{"id": ML, "min_level": 4.0}],
        },
        {
            "id": "analysis",
            "n_slots": 3,
            "skill_requirements": [{"id": DS, "min_level": 4.0}],
        },
    ],
}

r = requests.post(f"{BASE}/projects", json=project_payload)
r.raise_for_status()
project = r.json()
project_id = project["id"]
print(f"Created project id={project_id}")

Created project id=a9e43811-4de5-4855-b3e3-c6317d851e0e


## Solve

Same weights both times — only `handover` differs.

In [5]:
base_weights = {"performance": 0.5, "chemistry": 0.1, "growth": 0.2, "cost": 0.2}

r_independent = requests.post(f"{BASE}/optimization/solve", json={
    "project_id": project_id,
    "weights": {**base_weights, "handover": 0.0},
})
r_independent.raise_for_status()

r_handover = requests.post(f"{BASE}/optimization/solve", json={
    "project_id": project_id,
    "weights": {**base_weights, "handover": 0.4},
})
r_handover.raise_for_status()

result_independent = r_independent.json()
result_handover    = r_handover.json()

## Results

`retained` is the set of people kept across both phases — the metric `handover` is meant to maximise.

In [6]:
def show(result):
    print(f'Project {result["project_id"]}  (score={result["optimization_score"]})')
    by_phase = {}
    for m in result["members"]:
        name = id_to_name[m["person_id"]]
        by_phase.setdefault(m["phase_id"], []).append(name)
    for phase_id, names in by_phase.items():
        print(f'  {phase_id:<10} {sorted(names)}')
    retained = set.intersection(*(set(names) for names in by_phase.values()))
    print(f'  retained   {sorted(retained)}  ({len(retained)} kept across phases)')
    print()

print('=== handover = 0 (phases solved independently) ===')
show(result_independent)

print('=== handover = 0.4 (joint solve, team kept stable) ===')
show(result_handover)

=== handover = 0 (phases solved independently) ===
Project a9e43811-4de5-4855-b3e3-c6317d851e0e  (score=3.87)
  modeling   ['ml_1', 'ml_2', 'ml_3']
  analysis   ['ds_1', 'ds_2', 'ds_3']
  retained   []  (0 kept across phases)

=== handover = 0.4 (joint solve, team kept stable) ===
Project a9e43811-4de5-4855-b3e3-c6317d851e0e  (score=4.5225)
  modeling   ['ds_1', 'ds_5', 'ml_3']
  analysis   ['ds_1', 'ds_5', 'ml_3']
  retained   ['ds_1', 'ds_5', 'ml_3']  (3 kept across phases)



## Cleanup

Delete the people and project created by this notebook.

In [7]:
for person in created_people:
    requests.delete(f"{BASE}/people/{person['id']}").raise_for_status()

requests.delete(f"{BASE}/projects/{project_id}").raise_for_status()

print(f"Deleted {len(created_people)} people and project {project_id}")

Deleted 10 people and project a9e43811-4de5-4855-b3e3-c6317d851e0e
